In [10]:
def Divide(a,b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('不能除以零')
        return None
    except TypeError:
        print('参数类型错误')
        return None
    else:
        print(f'{a} / {b} = {result}')
        return result
    finally:
        print('清理工作完成')
        
print(Divide(10,0))
print(Divide(10,2))


不能除以零
清理工作完成
None
10 / 2 = 5.0
清理工作完成
5.0


In [12]:
def good_read_config(path):
    try:
        with open(path,encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f'未找到文件{path}')
        return 'default = true'
    except PermissionError:
        print('没有读取权限')
        raise
    except UnicodeDecodeError:
        print('编码错误')
        with open(path,encoding='GBK') as f:
            return f.read()

print(good_read_config('不存在文件.txt'))
            


未找到文件不存在文件.txt
default = true


In [16]:
# 最简单的自定义异常：继承 Exception 就行
class AppError(Exception):
    """项目基础异常（所有自定义异常的父类）"""
    pass

class ValidationError(AppError):
    """数据验证失败"""
    def __init__(self, field, value, message=""):
        self.field = field
        self.value = value
        super().__init__(message or f"字段 '{field}' 的值 '{value}' 不合法")

class NotFoundError(AppError):
    """资源不存在"""
    def __init__(self, resource, identifier):
        self.resource = resource
        self.identifier = identifier
        super().__init__(f"{resource} '{identifier}' 不存在")

class PermissionDeniedError(AppError):
    """权限不足"""
    pass

# 使用
def create_user(name, age, email):
    if not name or len(name) < 2:
        raise ValidationError("name", name, "用户名至少 2 个字符")
    if not (0 <= age <= 150):
        raise ValidationError("age", age, f"年龄必须在 0-150 之间")
    if "@" not in email:
        raise ValidationError("email", email, "邮箱格式不正确")
    return {"name": name, "age": age, "email": email}

# 测试
try:
    create_user("", 7, "xiaoming@test.com")
except ValidationError as e:
    print(f"❌ 验证失败: {e}")
    print(f"   问题字段: {e.field}")
    print(f"   问题值: {e.value}")
# ❌ 验证失败: 年龄必须在 0-150 之间
#    问题字段: age
#    问题值: -5

❌ 验证失败: 用户名至少 2 个字符
   问题字段: name
   问题值: 


In [17]:
class AppError(Exception):
    pass
class ValidationError(AppError):
    def __init__(self,field,value,message=''):
        self.field = field
        self.value = value
        super().__init__(message or f'字段{field}的{value}不合法')
class NotFoundError(AppError):
    def __init__(self,resource,identifier):
        self.resource = resource
        self.identifier = identifier
        super().__init__(f"'{resource}''{identifier}'不存在")
class PermissionError(AppError):
    pass

def create_user(name,age,email):
    if not name or len(name) < 2:
        raise ValidationError("name", name, "用户名至少 2 个字符")
    if not (0 <= age <= 150):
        raise ValidationError('age',age,'年龄不合法')
    if '@' not in email:
        raise ValidationError('email',email,'邮箱格式不正确')
    return {'name':name,'age':age,'email':email}

try:
   create_user('小明',15,'xiaomingtest.com')
except ValidationError as e:
    print(e)
    print(e.field)
    print(e.value) 


邮箱格式不正确
email
xiaomingtest.com


In [33]:
class ShopError(Exception):
    """电商系统基础异常"""
    pass
class ProductError(ShopError):
    """商品相关错误"""
    pass
class ProductNotFoundError(ProductError):
    def __init__(self,product_id):
        super().__init__(f'商品ID={product_id}不存在')
        self.prodcut_id = product_id
class OutOfStockError(ProductError):
    def __init__(self,product_name,stock):
        super().__init__(f"'{product_name}' 库存不足(剩余 {stock})")
        self.product_name = product_name
class OrderError(ShopError):
    """订单相关错误"""
    pass
class PaymentError(OrderError):
    """支付失败"""
    pass
class InsufficientBalanceError(PaymentError):
    def __init__(self,required,available):
        super().__init__(f"余额不足需要{required},仅有{available}")
        self.shortage = required - available

def buy(product_id,quantity,balance):
    products = {1:('机械键盘',599,3),2:('鼠标',129,0)}

    if  product_id not in products:
        raise ProductNotFoundError(product_id)
    name,price,stock = products[product_id]
    if stock < quantity:
        raise OutOfStockError(name,stock)
    total = price * quantity
    if balance < total:
        raise InsufficientBalanceError(total,balance)
    return f"✅ 购买成功: {name} x{quantity}，花费 ¥{total}"

try:
    buy(99,1,100)
except ProductNotFoundError as e:
    print(e)

try:
    buy(2,1,1000)
except OutOfStockError as e:
    print(e)

try:
    buy(1,2,500)
except InsufficientBalanceError as e:
    print(f"💰 {e}，还差 ¥{e.shortage}")

for pid in [1,2,99]:
    try:
        print(buy(pid,1,10000))
    except ProductError as e:
        print(e)
    except ShopError as e:
        print(e)

商品ID=99不存在
'鼠标' 库存不足(剩余 0)
💰 余额不足需要1198,仅有500，还差 ¥698
✅ 购买成功: 机械键盘 x1，花费 ¥599
'鼠标' 库存不足(剩余 0)
商品ID=99不存在
